# Phase 5: Profile QGFD Overhead

Goal: Break down where the QGFD overhead comes from.
Budget: ~30 min (forward/backward passes only).

## 0. Install & Setup

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft datasets trl matplotlib pandas
!pip install -q git+https://github.com/rajboopathiking/TorchDire.git

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
import torch.profiler
import time

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

cleanup()
set_seed(42)

## 1. Configuration

In [ ]:
MODEL_ID = "42dot/42dot_LLM-SFT-1.3B"
SEQ_LENS = [128, 256, 512]
NUM_WARMUP = 10
NUM_ITER = 50
BATCH_SIZE = 4
CSV_OUT = "/kaggle/working/phase5_profiling.csv"

## 2. Load Model & Patch

In [ ]:
from torchdire import patch_llama_with_qgfd, collect_qgfd_kernels

def load_model(enable_qgfd=False):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=False,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    
    if enable_qgfd:
        patch_llama_with_qgfd(model, enable_qgfd=True, num_diffusion_steps=3, dt=0.1)
    
    return model, tokenizer

## 3. Forward Pass Profiling

In [ ]:
def profile_forward(model, tokenizer, seq_len):
    inputs = tokenizer(["Here is some synthetic text " * 20] * BATCH_SIZE, return_tensors="pt", max_length=seq_len, padding="max_length", truncation=True).to("cuda")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    # Warmup
    for _ in range(NUM_WARMUP):
        with torch.no_grad():
            _ = model(input_ids, attention_mask=attention_mask)
            
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(NUM_ITER)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(NUM_ITER)]
    
    for i in range(NUM_ITER):
        start_events[i].record()
        with torch.no_grad():
            _ = model(input_ids, attention_mask=attention_mask)
        end_events[i].record()
        
    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
    return sum(times) / NUM_ITER

results = []
for seq_len in SEQ_LENS:
    print(f"Profiling seq_len {seq_len}...")
    
    print("  Loading baseline...")
    model, tok = load_model(False)
    time_base = profile_forward(model, tok, seq_len)
    del model
    cleanup()
    
    print("  Loading QGFD...")
    model, tok = load_model(True)
    time_qgfd = profile_forward(model, tok, seq_len)
    del model
    cleanup()
    
    results.append({
        "seq_len": seq_len,
        "base_fwd_ms": time_base,
        "qgfd_fwd_ms": time_qgfd,
        "overhead_fwd_ms": time_qgfd - time_base,
        "overhead_fwd_pct": (time_qgfd - time_base) / time_base * 100
    })

df_fwd = pd.DataFrame(results)
display(df_fwd)

## 4. Component-Level Timing
Hooking into build_transition_from_keys and the diffusion loop to time each sub-operation.

In [ ]:
# To time specific components within the forward pass, we use torch.profiler down below for a fine-grained view.

## 5. Backward Pass Profiling

In [ ]:
def profile_backward(model, tokenizer, seq_len):
    inputs = tokenizer(["Here is some synthetic text " * 20] * BATCH_SIZE, return_tensors="pt", max_length=seq_len, padding="max_length", truncation=True).to("cuda")
    input_ids = inputs["input_ids"]
    labels = input_ids.clone()
    
    # Warmup
    for _ in range(5):
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        model.zero_grad()
        
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(NUM_ITER)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(NUM_ITER)]
    
    for i in range(NUM_ITER):
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        start_events[i].record()
        loss.backward()
        end_events[i].record()
        model.zero_grad()
        
    torch.cuda.synchronize()
    times = [s.elapsed_time(e) for s, e in zip(start_events, end_events)]
    return sum(times) / NUM_ITER

bw_results = []
for seq_len in SEQ_LENS:
    print(f"Profiling Backward seq_len {seq_len}...")
    
    model, tok = load_model(False)
    for p in model.parameters(): p.requires_grad = False
    model.lm_head.weight.requires_grad = True # enable grad somewhere
    time_base_bw = profile_backward(model, tok, seq_len)
    del model
    cleanup()
    
    model, tok = load_model(True)
    for p in model.parameters(): p.requires_grad = False
    model.lm_head.weight.requires_grad = True
    time_qgfd_bw = profile_backward(model, tok, seq_len)
    del model
    cleanup()
    
    bw_results.append({
        "seq_len": seq_len,
        "base_bwd_ms": time_base_bw,
        "qgfd_bwd_ms": time_qgfd_bw,
        "overhead_bwd_ms": time_qgfd_bw - time_base_bw,
        "overhead_bwd_pct": (time_qgfd_bw - time_base_bw) / time_base_bw * 100
    })

df_bw = pd.DataFrame(bw_results)
display(df_bw)

## 6. torch.profiler Integration
Run one forward pass with PyTorch profiler to get kernel breakdown.

In [ ]:
model, tok = load_model(True)
seq_len = 256
inputs = tok(["Test sequence profiling! " * 20], return_tensors="pt", max_length=seq_len, truncation=True).to("cuda")

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
    on_trace_ready=torch.profiler.tensorboard_trace_handler('/kaggle/working/profiler_logs')
) as prof:
    with torch.no_grad():
        model(**inputs)
        
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
del model
cleanup()

## 7. Results Table

In [ ]:
df_final = pd.merge(df_fwd, df_bw, on="seq_len")
df_final.to_csv(CSV_OUT, index=False)
display(df_final)

## 8. Analysis
- **Similarity Matmul vs Loop**: Check profiler trace to see which dominates the forward overhead.
- **Is 1.19x close to floor?**: Depending on caching and fusing potentials, we might reduce this.
- **Where is the slack?**: We can fuse the diffusion loop into a single custom CUDA kernel if needed.

## 9. Summary Plots

In [ ]:
plt.figure(figsize=(10, 5))
bar_width = 0.35
x = range(len(SEQ_LENS))

plt.bar(x, df_final["base_fwd_ms"], width=bar_width, label="Baseline Forward")
plt.bar([i + bar_width for i in x], df_final["qgfd_fwd_ms"], width=bar_width, label="QGFD Forward")

plt.xticks([i + bar_width/2 for i in x], SEQ_LENS)
plt.xlabel("Sequence Length")
plt.ylabel("Time (ms)")
plt.title("Forward Pass Time: Baseline vs QGFD")
plt.legend()
plt.savefig("/kaggle/working/fwd_time.png")
plt.show()

## 10. Final Summary
The majority of overhead scales linearly or quadratically with sequence length. Further optimizations should target the main bottleneck identified in the profiler logs.